# 05 – Un piccolo agente

Un agente è un **LLM che può usare strumenti**, dentro un **ciclo**:

1. **pensa**: il modello decide se gli serve uno strumento;
2. **agisce**: il nostro programma esegue lo strumento;
3. **osserva**: il risultato torna al modello;
4. **ripete**, finché il modello può rispondere.

Usiamo **Qwen2.5 3B Instruct**, che gira gratis in Colab (la prima volta scarica circa 6 GB, qualche minuto). È piccolo: a volte sbaglia, e anche questo è istruttivo.

Prima di iniziare: **Runtime → Cambia tipo di runtime → T4 GPU**.

In [ ]:
import json, re, ast, operator
import pandas as pd
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # evita il download che in Colab a volte si blocca
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

nome = 'Qwen/Qwen2.5-3B-Instruct'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(nome)
modello = AutoModelForCausalLM.from_pretrained(nome, torch_dtype='auto').to(dispositivo)
print('Pronto su', dispositivo)

## 1. I dati: una piccola tabella di ordini

In [ ]:
ordini = pd.DataFrame({
    'numero':  ['ORD-101', 'ORD-102', 'ORD-103', 'ORD-104', 'ORD-105', 'ORD-106'],
    'cliente': ['Rossi', 'Bianchi', 'Rossi', 'Verdi', 'Neri', 'Bianchi'],
    'stato':   ['aperto', 'evaso', 'aperto', 'aperto', 'annullato', 'evaso'],
    'righe':   [12, 4, 7, 30, 2, 9],
})
ordini

## 2. Gli strumenti

Sono normali funzioni Python. La **docstring** è importante: è la descrizione che il modello legge per decidere quando usarle.

In [ ]:
def cerca_ordini(stato: str = None, cliente: str = None) -> str:
    """
    Cerca gli ordini nel gestionale, filtrando per stato e/o cliente.

    Args:
        stato: stato dell'ordine: aperto, evaso oppure annullato
        cliente: cognome del cliente, per esempio Rossi
    """
    r = ordini
    if stato:
        r = r[r.stato == stato.lower()]
    if cliente:
        r = r[r.cliente.str.lower() == cliente.lower()]
    return r.to_json(orient='records', force_ascii=False)


_OP = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.USub: operator.neg}

def calcolatrice(espressione: str) -> str:
    """
    Calcola un'espressione aritmetica con + - * / e parentesi.

    Args:
        espressione: l'espressione da calcolare, per esempio (12 + 7) * 2
    """
    def valuta(n):
        if isinstance(n, ast.Constant):
            return n.value
        if isinstance(n, ast.BinOp):
            return _OP[type(n.op)](valuta(n.left), valuta(n.right))
        if isinstance(n, ast.UnaryOp):
            return _OP[type(n.op)](valuta(n.operand))
        raise ValueError('espressione non permessa')
    return str(valuta(ast.parse(espressione, mode='eval').body))


strumenti = {'cerca_ordini': cerca_ordini, 'calcolatrice': calcolatrice}
print(cerca_ordini(stato='aperto'))
print(calcolatrice('(12 + 7) * 2'))

## 3. Che cosa vede il modello

Il testo che arriva al modello contiene la descrizione degli strumenti.
Il modello **non esegue niente**: se vuole uno strumento, scrive un blocco `<tool_call>` con nome e argomenti.

In [ ]:
messaggi = [{'role': 'user', 'content': 'Quanti ordini aperti ci sono?'}]
prompt = tokenizer.apply_chat_template(messaggi, tools=list(strumenti.values()),
                                       tokenize=False, add_generation_prompt=True)
print(prompt)

## 4. Il ciclo dell'agente

In [ ]:
SISTEMA = ('Sei l\'assistente del magazzino. Rispondi in italiano. '
           'Per i dati sugli ordini usa sempre lo strumento cerca_ordini, '
           'per i conti usa la calcolatrice. Non inventare dati.')

def llm(messaggi):
    testo = tokenizer.apply_chat_template(messaggi, tools=list(strumenti.values()),
                                          tokenize=False, add_generation_prompt=True)
    ids = tokenizer(testo, return_tensors='pt').input_ids.to(dispositivo)
    out = modello.generate(ids, max_new_tokens=300, do_sample=False, repetition_penalty=1.1,
                           pad_token_id=tokenizer.eos_token_id)
    risposta = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=False)
    return risposta.replace('<|im_end|>', '').replace('<|endoftext|>', '').strip()


def agente(domanda, passi_max=5):
    messaggi = [{'role': 'system', 'content': SISTEMA}, {'role': 'user', 'content': domanda}]
    print('DOMANDA:', domanda)
    for passo in range(1, passi_max + 1):
        risposta = llm(messaggi)
        chiamate = re.findall(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', risposta, re.S)
        if not chiamate:
            print(f'\n[{passo}] RISPOSTA:', risposta)
            return risposta
        try:
            chiamate = [json.loads(c) for c in chiamate]
        except json.JSONDecodeError:
            print(f'\n[{passo}] Il modello ha scritto una chiamata non valida:', risposta)
            messaggi.append({'role': 'assistant', 'content': risposta})
            messaggi.append({'role': 'user', 'content': 'La chiamata allo strumento non è JSON valido: riprova.'})
            continue
        messaggi.append({'role': 'assistant', 'content': '',
                         'tool_calls': [{'type': 'function', 'function': c} for c in chiamate]})
        for c in chiamate:
            print(f'\n[{passo}] PENSA → usa {c.get("name")} con {c.get("arguments")}')
            try:
                risultato = strumenti[c['name']](**c['arguments'])
            except Exception as errore:
                risultato = f'Errore: {errore}'
            print(f'[{passo}] OSSERVA →', risultato)
            messaggi.append({'role': 'tool', 'name': c['name'], 'content': risultato})
    print('Troppi passi: mi fermo.')

In [ ]:
agente('Quanti ordini aperti ci sono?')

In [ ]:
agente('Quante righe in totale hanno gli ordini aperti del cliente Rossi?')

In [ ]:
agente('Che tempo fa a Napoli?')

L'ultima domanda non ha uno strumento adatto: osserva se il modello lo ammette o inventa.

## 5. Prova tu

1. Fai una domanda tua sugli ordini.
2. Aggiungi uno strumento, per esempio `annulla_ordine(numero)` che cambia lo stato nella tabella,
   aggiungilo al dizionario `strumenti` e chiedi all'agente di annullare un ordine.
3. Domanda per tutti: uno strumento che **modifica** i dati dovrebbe chiedere conferma prima di agire?

In [ ]:
agente('scrivi qui la tua domanda')

## Da ricordare

- Agente = LLM + strumenti + ciclo.
- L'LLM sceglie lo strumento; è il programma che lo esegue.
- La descrizione degli strumenti è parte del prompt: scriverla bene conta.
- Più potere agli strumenti = più bisogno di permessi e controlli.